In [ ]:
! pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


In [ ]:
data = [
 {
        "instruction":"What is AI?",
        "response":"Artificial Intelligence is the simulation of human intelligence by machines."
    },{
        "instruction":"What is Machine Learning?",
        "response":"Machine Learning is a subset of Artificial Intelligence."
    },{
        "instruction":"What is Deep Learning?",
        "response":"Deep Learning uses neural networks with many hidden layers."
    },{
        "instruction":"Who developed Python?",
        "response":"Python was developed by Guido van Rossum."
    },
 {
        "instruction":"What is NLP?",
        "response":"Natural Language Processing enables computers to understand human language."
    }]
print(data)

[{'instruction': 'What is AI?', 'response': 'Artificial Intelligence is the simulation of human intelligence by machines.'}, {'instruction': 'What is Machine Learning?', 'response': 'Machine Learning is a subset of Artificial Intelligence.'}, {'instruction': 'What is Deep Learning?', 'response': 'Deep Learning uses neural networks with many hidden layers.'}, {'instruction': 'Who developed Python?', 'response': 'Python was developed by Guido van Rossum.'}, {'instruction': 'What is NLP?', 'response': 'Natural Language Processing enables computers to understand human language.'}]


In [ ]:
# 3 CONVERT INTO HUGGIN FACE DATASET - BCZ PYTHON LIST IS NOT COMPATIBLE
from datasets import Dataset

dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 5
})


In [ ]:
print(dataset[0])

{'instruction': 'What is AI?', 'response': 'Artificial Intelligence is the simulation of human intelligence by machines.'}


In [ ]:
# 5 fomrat columns : convert the entire thing to text - Datasets format to text


def format_data(example):
  text = f'''
   ## instruction : {example['instruction']}

   ## response : {example['response']}
  '''
  example['text'] = text
  return example

dataset = dataset.map(format_data)
print(dataset['text'][0])


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

 
   ## instruction : What is AI?

   ## response : Artificial Intelligence is the simulation of human intelligence by machines.
  


In [ ]:
# 7 load tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# 8 set PAD tokens, gpt 2 doenst have padding , we use EOS instead of <Pad>


tokenizer.pad_token = tokenizer.eos_token




In [ ]:
def tokenize(example):
  return tokenizer(
      example['text'],
      truncation = True,
      padding = 'max_length',
      max_length = 128

  )

tokenized_dataset = dataset.map(tokenize)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset[0]['input_ids'])

[220, 198, 220, 220, 22492, 12064, 1058, 1867, 318, 9552, 30, 628, 220, 220, 22492, 2882, 1058, 35941, 9345, 318, 262, 18640, 286, 1692, 4430, 416, 8217, 13, 198, 220, 220, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]


In [ ]:
from transformers import AutoModelForCausalLM


model =AutoModelForCausalLM.from_pretrained('gpt2')

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# create labels for causal language modeling
# In causal language modeling, the model predicts the next token in a sequence.
# Therefore, the input sequence itself serves as the target labels for training.
def add_labels(example):
  example['labels'] = example['input_ids'].copy()

  return example

tokenized_dataset = tokenized_dataset.map(add_labels)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset)

Dataset({
    features: ['instruction', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = './model',
    num_train_epochs = 10 ,
    per_device_train_batch_size = 2,
    learning_rate = 5e-5,
    logging_steps = 1,
    save_strategy = 'epoch',
    report_to = 'none'

)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset,

)

In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,8.029940
2,3.579022
3,1.782256
4,1.169411
5,1.301887
6,1.163925
7,1.209816
8,1.038936
9,1.035000
10,0.861068


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=1.0434412638346353, metrics={'train_runtime': 139.7103, 'train_samples_per_second': 0.358, 'train_steps_per_second': 0.215, 'total_flos': 3266150400000.0, 'train_loss': 1.0434412638346353, 'epoch': 10.0})

In [ ]:
trainer.save_model('my_model')
tokenizer.save_pretrained('my_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_model/tokenizer_config.json', 'my_model/tokenizer.json')

In [ ]:
# 17  test the model

from transformers import pipeline

generator = pipeline('text-generation',
                     model = 'my_model',
                     tokenizer = 'my_model')

prompt = ''' ## instruction : who developed python ?  '''

output = generator(
    prompt,
    max_new_tokens = 30
)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 ## instruction : who developed python ?    ## response : python is a language developed by humans.  
